# 01 — Data Ingestion and Cleaning

Load all raw datasets, clean and standardize them, convert coordinate systems to WGS84 (EPSG:4326), and save cleaned versions to `data/processed/`.

**Datasets loaded:**
1. Road network (Ministry of Transport)
2. Existing EV chargers (NAP — National Access Point)
3. Grid capacity (i-DE, Endesa, Viesgo)
4. EV registration forecast data (datos.gob.es parquets)
5. IMD traffic count stations (DGT)
6. Gas stations (MITECO)
7. Population (INE)
8. Tourism seasonality (INE)
9. Service areas (OSM/Hermes)
10. TEN-T corridors
11. Administrative boundaries (provinces, CCAA)

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
# If running from notebooks/ folder, adjust path to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import geopandas as gpd
from pathlib import Path

from src.data_loading import (
    load_road_network, load_nap_charging_points, load_grid_capacity,
    load_ev_forecast, load_imd_traffic, load_gas_stations,
    load_population, load_tourism_seasonal, load_service_areas,
    load_tent_corridors, load_provinces, load_ccaa,
)

PROCESSED = Path('data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Setup complete. Project root:', os.getcwd())

## 1. Road Network (Ministry of Transport)

In [ ]:
roads = load_road_network()
print(f"Road network: {roads.shape[0]} segments, CRS: {roads.crs}")
print(f"Columns: {roads.columns.tolist()}")
print(f"\nRoad type distribution:")
print(roads['Tipo_de_via'].value_counts())

# Clean: drop segments without road name, validate geometry
roads = roads.dropna(subset=['Carretera'])
roads['geometry'] = roads.geometry.make_valid()

# Add useful columns
roads['length_km'] = roads['Longitud'] / 1000.0
roads['road_prefix'] = roads['Carretera'].str.extract(r'^(AP|A|N|[A-Z]+)')

# Save
roads.to_file(PROCESSED / 'roads_clean.geojson', driver='GeoJSON')
print(f"\nSaved: roads_clean.geojson ({roads.shape[0]} segments)")
roads.head()

## 2. Existing EV Charging Points (NAP)

Parse the 83 MB DATEX II XML file using streaming `lxml.etree.iterparse`. This extracts ~12,000 charging sites with coordinates, connector count, and max power.

In [ ]:
chargers = load_nap_charging_points()
print(f"NAP Charging Points: {chargers.shape[0]} sites")
print(f"Lat range: {chargers.latitude.min():.2f} – {chargers.latitude.max():.2f}")
print(f"Lon range: {chargers.longitude.min():.2f} – {chargers.longitude.max():.2f}")
print(f"Max power (kW): {chargers.max_power_kw.describe()}")
print(f"\nTop provinces:")
print(chargers['province'].value_counts().head(10))

# Validate coordinates are within Spain
chargers = chargers[
    (chargers.latitude >= 27) & (chargers.latitude <= 44) &
    (chargers.longitude >= -19) & (chargers.longitude <= 5)
]
chargers = chargers.drop_duplicates(subset=['site_id'])

# Save
chargers.to_csv(PROCESSED / 'chargers_clean.csv', index=False)
print(f"\nSaved: chargers_clean.csv ({chargers.shape[0]} sites)")
chargers.head()

## 3. Grid Capacity (i-DE, Endesa, Viesgo)

Load substation-level capacity data from three electricity distributors. Each has different file encoding, column names, and number formats. UTM coordinates (EPSG:25830) are converted to WGS84.

**Key caveat:** Endesa files are "generación" (generation capacity), not "demanda" — we use available capacity as a proxy for grid headroom. This is documented in assumptions.md.

In [ ]:
grid = load_grid_capacity('all')
print(f"Grid Capacity: {grid.shape[0]} rows")
print(f"\nDistributor breakdown:")
print(grid['distributor_network'].value_counts())
print(f"\nCapacity stats (MW):")
print(grid['available_capacity_mw'].describe())
print(f"\nCoordinate ranges:")
print(f"  Lat: {grid.latitude.min():.2f} – {grid.latitude.max():.2f}")
print(f"  Lon: {grid.longitude.min():.2f} – {grid.longitude.max():.2f}")
print(f"\nProvinces covered: {grid.provincia.nunique()}")

# Save
grid.to_csv(PROCESSED / 'grid_capacity_unified.csv', index=False)
print(f"\nSaved: grid_capacity_unified.csv ({grid.shape[0]} rows)")
grid.head()

## 4. EV Registration Forecast Data (datos.gob.es)

Load 108 monthly parquet files (2015–2023) of vehicle registration microdata. Filter to new BEV+PHEV registrations and aggregate to a monthly time series. This feeds the SARIMA model in Notebook 02.

In [ ]:
ev_ts = load_ev_forecast()
print(f"EV Registration Time Series: {ev_ts.shape[0]} months")
print(f"Date range: {ev_ts.date.min()} – {ev_ts.date.max()}")
print(f"Total BEV+PHEV registrations (2015–2023): {ev_ts.ev_registrations.sum():,}")
print(f"\nMonthly stats:")
print(ev_ts['ev_registrations'].describe())

# Save
ev_ts.to_csv(PROCESSED / 'ev_monthly_registrations.csv', index=False)
print(f"\nSaved: ev_monthly_registrations.csv ({ev_ts.shape[0]} rows)")
ev_ts.tail(12)

## 5. IMD Traffic Count Stations (DGT)

In [ ]:
imd = load_imd_traffic()
print(f"IMD Traffic Stations: {imd.shape[0]} stations, CRS: {imd.crs}")
print(f"Columns: {imd.columns.tolist()}")
print(f"\nIMD total stats:")
if 'IMD_total' in imd.columns:
    print(imd['IMD_total'].describe())

# Save
imd.to_file(PROCESSED / 'imd_traffic_clean.geojson', driver='GeoJSON')
print(f"\nSaved: imd_traffic_clean.geojson ({imd.shape[0]} stations)")
imd.head()

## 6. Supplementary Datasets

Gas stations, population, tourism, service areas, TEN-T corridors, and administrative boundaries.

In [ ]:
# Gas stations (MITECO) — potential candidate locations for co-location
gas = load_gas_stations()
print(f"Gas Stations: {gas.shape[0]} stations")
gas.to_csv(PROCESSED / 'gas_stations_clean.csv', index=False)
print(f"Saved: gas_stations_clean.csv")

# Population (INE)
pop = load_population()
print(f"\nPopulation: {pop.shape[0]} municipalities")
pop.to_csv(PROCESSED / 'population_municipal.csv', index=False)
print(f"Saved: population_municipal.csv")

# Tourism seasonality (INE)
tourism = load_tourism_seasonal()
print(f"\nTourism: {tourism.shape[0]} rows")
tourism.to_csv(PROCESSED / 'tourism_seasonal.csv', index=False)
print(f"Saved: tourism_seasonal.csv")

# Service areas (OSM/Hermes)
svc = load_service_areas()
print(f"\nService Areas: {svc.shape[0]} areas")
svc.to_file(PROCESSED / 'service_areas_clean.geojson', driver='GeoJSON')
print(f"Saved: service_areas_clean.geojson")

# TEN-T corridors
tent = load_tent_corridors()
print(f"\nTEN-T Corridors: {tent.shape[0]} segments")
tent.to_file(PROCESSED / 'tent_corridors.geojson', driver='GeoJSON')
print(f"Saved: tent_corridors.geojson")

# Province and CCAA boundaries
provinces = load_provinces()
provinces.to_file(PROCESSED / 'provinces.geojson', driver='GeoJSON')
print(f"\nProvinces: {provinces.shape[0]} → saved")

ccaa = load_ccaa()
ccaa.to_file(PROCESSED / 'ccaa.geojson', driver='GeoJSON')
print(f"CCAA: {ccaa.shape[0]} → saved")

## Summary — All Processed Files

In [ ]:
import os

summary = []
for f in sorted(PROCESSED.glob('*')):
    size_kb = os.path.getsize(f) / 1024
    summary.append({'file': f.name, 'size_kb': round(size_kb, 1)})

summary_df = pd.DataFrame(summary)
print(f"Total processed files: {len(summary_df)}")
print(f"Total size: {summary_df['size_kb'].sum() / 1024:.1f} MB")
summary_df